# YOLOv8 Retraining: Crosswalk Pedestrian Detection
### Hardware Profile: Acer Nitro ANV15-41 (NVIDIA RTX 2050 4GB / 8GB RAM)

- **Dataset:** 1,500 labeled frames (1,200 train / 300 val) from `datasets/august11_15_Batch_Yolo/`  
- **Classes:** `adult (0)`, `child (1)`  
- **Starting Checkpoint:** `models/yolov8n_crosswalk_v2_1111_frames.pt` (fine-tuning)  
- **Tuned Parameters:** `imgsz=640`, `batch=8` (optimal for 4GB VRAM), `workers=0` (Windows), `device=0` (RTX 2050)

In [ ]:
# Cell 1: Environment & GPU Verification
import ultralytics
import torch
from ultralytics import YOLO

ultralytics.checks()

# Confirm PyTorch detects your NVIDIA GeForce RTX 2050
if torch.cuda.is_available():
    print(f'CUDA Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total:  {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    print('[WARNING] CUDA not detected. Training will fall back to CPU.')

## 🚀 Start Fine-Tuning Model
Executes YOLOv8 training on your local RTX 2050 GPU via the connected Colab/Jupyter local runtime.

In [ ]:
# Cell 2: Train Model
model = YOLO('models/yolov8n_crosswalk_v2_1111_frames.pt')

DATASET_YAML = r'C:/Users/Acer/Documents/CAPSTONE/web_dashboard_analytics/datasets/august11_15_Batch_Yolo/data.yaml'
PROJECT_DIR  = r'C:/Users/Acer/Documents/CAPSTONE/web_dashboard_analytics/datasets/03_training_runs'

results = model.train(
    data=DATASET_YAML,
    epochs=100,
    imgsz=640,
    batch=8,            # Tuned specifically for RTX 2050 4GB VRAM
    workers=0,          # Mandatory on Windows to prevent process deadlocks
    device=0,           # Forces dedicated NVIDIA GeForce RTX 2050
    cache=True,         # Caches images in RAM for high-speed epochs
    project=PROJECT_DIR,
    name='run_august11_15batches',
    exist_ok=True,
    patience=20,        # Early stopping if no improvement for 20 epochs
    augment=True,
    cos_lr=True,        # Cosine learning rate decay
    close_mosaic=20     # Disables mosaic during final 20 epochs for fine detail
)

## 📊 Validate Best Checkpoint
Computes mAP50, mAP50-95, precision, and recall on the validation set using the best weights.

In [ ]:
# Cell 3: Validation Metrics
best_pt = r'C:/Users/Acer/Documents/CAPSTONE/web_dashboard_analytics/datasets/03_training_runs/run_august11_15batches/weights/best.pt'
trained_model = YOLO(best_pt)
metrics = trained_model.val()

print(f'Validation mAP50:    {metrics.box.map50:.5f}')
print(f'Validation mAP50-95: {metrics.box.map:.5f}')